# Groupby and Aggregation

## Introduction

Splitting a dataset into subgroups and computing summary statistics for each group is one of the most common operations in exploratory data analysis. Pandas makes this concise with `.groupby()`.

## Objectives

You will be able to:

- Use `.groupby()` to split a DataFrame into subgroups
- Chain aggregation functions (`.mean()`, `.sum()`, `.count()`, etc.) onto a grouped object
- Group by multiple columns to create multi-level aggregations
- Select specific columns from a grouped object and slice results by index

In [ ]:
import pandas as pd

df = pd.read_csv('data/dealing_missing_data_lab/titanic.csv', index_col=0)
df.drop(columns='Cabin', inplace=True, errors='ignore')
df['Age'].fillna(df['Age'].median(), inplace=True)
df.dropna(subset=['Embarked'], inplace=True)

print(df.shape)
df.head()

---

## The `.groupby()` Method

`.groupby()` splits the DataFrame into subgroups by the values in one or more columns:

```python
df.groupby('Sex')
```

This alone returns a `GroupBy` object — not a DataFrame. It becomes useful when you chain an **aggregation function** onto it:

```python
df.groupby('Sex').sum()
```

![groupby sum by sex](assets/pandas_groupby/titanic_2.png)

In [ ]:
df.groupby('Sex').sum(numeric_only=True)

---

## Aggregation Functions

All of the following can be chained onto a grouped object:

| Method | Returns |
|--------|---------|
| `.min()` | Minimum value per group |
| `.max()` | Maximum value per group |
| `.mean()` | Mean per group |
| `.median()` | Median per group |
| `.sum()` | Sum per group |
| `.count()` | Non-null count per group |
| `.std()` | Standard deviation per group |
| `.agg(['mean', 'std'])` | Multiple aggregations at once |

In [ ]:
# Survival rate by sex
df.groupby('Sex')['Survived'].mean().round(3)

In [ ]:
# Passenger count and mean age by sex
df.groupby('Sex').agg(
    count=('PassengerId', 'count'),
    mean_age=('Age', 'mean')
).round(1)

---

## Multiple Groups

Pass a list of column names to group by every combination of those values:

```python
df.groupby(['Sex', 'Pclass']).mean()
```

![groupby sex and pclass](assets/pandas_groupby/titanic_3.png)

In [ ]:
df.groupby(['Sex', 'Pclass']).mean(numeric_only=True).round(2)

---

## Selecting Columns from a Grouped Object

Slice a specific column **between** `.groupby()` and the aggregation to get a focused result:

```python
df.groupby(['Sex', 'Pclass'])['Survived'].mean()
```

![survival rate by sex and pclass](assets/pandas_groupby/titanic_4.png)

In [ ]:
survival = df.groupby(['Sex', 'Pclass'])['Survived'].mean().round(3)
survival

In [ ]:
# The result has a MultiIndex — slice by the outer level to get all female rows
print(survival['female'])

# Slice by both levels to get a single value
print(f"\nFemale, 1st class survival rate: {survival['female']['1']:.3f}")

---

## Practice: Titanic Investigations

Use `.groupby()` to answer the following questions about the dataset.

In [ ]:
# What was the average fare paid by each passenger class?
df.groupby('Pclass')['Fare'].mean().round(2)

In [ ]:
# What was the survival rate by passenger class?
df.groupby('Pclass')['Survived'].mean().round(3)

In [ ]:
# How many passengers embarked from each port?
df.groupby('Embarked')['PassengerId'].count()

In [ ]:
# Median age by sex and survival status — did survivors tend to be younger?
df.groupby(['Sex', 'Survived'])['Age'].median().round(1)

In [ ]:
# Visualise survival rate by class and sex together
(
    df.groupby(['Pclass', 'Sex'])['Survived']
    .mean()
    .unstack()          # Sex becomes columns
    .plot(kind='bar', figsize=(7, 4), title='Survival Rate by Class and Sex')
)
import matplotlib.pyplot as plt
plt.ylabel('Survival rate')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

---

## Summary

In this notebook you learned how to:

- Split a DataFrame with `.groupby()` and chain aggregation functions onto the result
- Use `.agg()` to compute multiple statistics in one call
- Group by multiple columns for cross-tabulation
- Slice a column from a grouped object and navigate the resulting MultiIndex

Next: [05 — Pivot Tables](05_pivot_tables.ipynb)